## Summary & Next Steps

### What We Accomplished

1. ✅ Loaded and explored the Joshi et al. (2016) Book Snippets dataset
2. ✅ Applied spaCy NER preprocessing to remove character names
3. ✅ Loaded the pre-trained `cardiffnlp/twitter-roberta-base-irony` model
4. ✅ Prepared data with stratified train/val/test splits
5. ✅ Implemented best practices for fine-tuning transformers
6. ✅ Trained the model with early stopping and F1-score monitoring
7. ✅ Evaluated performance on held-out test set
8. ✅ Created inference function for custom text

### Next Steps

**Production Deployment:**
- Deploy the fine-tuned model using the `TransformerSarcasmDetector` class
- Integrate with Streamlit app: `detector = TransformerSarcasmDetector(model_name="./models/fine_tuned_sarcasm")`

**Hyperparameter Tuning:**
- Experiment with learning rates: `[1e-5, 2e-5, 3e-5, 5e-5]`
- Try different batch sizes: `[4, 8, 16, 32]`
- Adjust epochs based on early stopping results

**Data Augmentation:**
- Mix Joshi data with other sarcasm datasets for robustness
- Use back-translation to generate synthetic examples
- Apply random word swaps for regularization

**Evaluation:**
- Test on out-of-domain datasets (SemEval, Twitter)
- Analyze model errors to identify weak patterns
- Compare with rule-based and other ML approaches

**Further Research:**
- Fine-tune larger models (RoBERTa-large, ELECTRA) if compute available
- Experiment with domain-adaptive pre-training on fiction text
- Implement confidence calibration for better uncertainty estimates

In [ ]:
# Test examples
test_examples = [
    ("Oh sure, because that always works perfectly.", True),
    ("I absolutely love waiting in traffic jams.", True),
    ("Yeah right, that makes total sense.", True),
    ("The weather is beautiful today.", False),
    ("I sincerely appreciate your kind gesture.", False),
    ("The sun is shining brightly in the sky.", False),
]

print("Testing Fine-tuned Model on Custom Examples\n")
print("=" * 80)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

for text, expected_label in test_examples:
    result = predict_sarcasm(text, model, tokenizer, nlp, device=device)
    
    expected_str = "Sarcastic" if expected_label else "Non-Sarcastic"
    predicted_str = "Sarcastic" if result["is_sarcastic"] else "Non-Sarcastic"
    correct = "✓" if (result["is_sarcastic"] == expected_label) else "✗"
    
    print(f"\nText: {result['text']}")
    print(f"Processed: {result['processed_text']}")
    print(f"Expected: {expected_str} | Predicted: {predicted_str} {correct}")
    print(f"Confidence: {result['confidence']:.1%}")
    print(f"Probabilities: Non-sarcastic={result['class_probabilities']['non_sarcastic']:.3f}, "
          f"Sarcastic={result['class_probabilities']['sarcastic']:.3f}")
    print("-" * 80)

## 10. Test on Custom Examples

Test the fine-tuned model on various sarcastic and non-sarcastic text.

In [ ]:
# Create inference function
def predict_sarcasm(text: str, model, tokenizer, nlp, device="cpu"):
    """Predict sarcasm for a given text."""
    
    # Preprocess (remove character names)
    doc = nlp(text)
    entities_to_replace = [
        (ent.start_char, ent.end_char)
        for ent in doc.ents
        if ent.label_ == "PERSON"
    ]
    
    processed_text = text
    for start, end in sorted(entities_to_replace, reverse=True):
        processed_text = processed_text[:start] + "[PERSON]" + processed_text[end:]
    
    # Tokenize
    inputs = tokenizer(
        processed_text,
        return_tensors="pt",
        truncation=True,
        max_length=512,
        padding=True
    )
    
    # Move to device
    inputs = {k: v.to(device) for k, v in inputs.items()}
    
    # Predict
    model.eval()
    with torch.no_grad():
        outputs = model(**inputs)
    
    logits = outputs.logits
    probs = torch.softmax(logits, dim=-1)
    pred_class = torch.argmax(logits, dim=-1).item()
    confidence = probs[0, pred_class].item()
    
    return {
        "text": text,
        "processed_text": processed_text,
        "is_sarcastic": bool(pred_class),
        "confidence": confidence,
        "class_probabilities": {
            "non_sarcastic": float(probs[0, 0].item()),
            "sarcastic": float(probs[0, 1].item())
        }
    }

print("✓ Inference function created")

In [ ]:
# Save the model
model_save_path = "./models/fine_tuned_sarcasm"
trainer.save_model(model_save_path)
tokenizer.save_pretrained(model_save_path)

print(f"✓ Model saved to: {model_save_path}")

## 9. Save and Test the Fine-tuned Model

Save the model and create a simple inference function to test on custom examples.

In [ ]:
# Visualize confusion matrix
fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=["Non-Sarcastic", "Sarcastic"],
            yticklabels=["Non-Sarcastic", "Sarcastic"],
            ax=ax, cbar_kws={'label': 'Count'})
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
ax.set_title("Test Set Confusion Matrix")
plt.tight_layout()
plt.show()

In [ ]:
# Get predictions for detailed analysis
predictions = trainer.predict(test_dataset)
pred_labels = np.argmax(predictions.predictions, axis=1)

# Confusion matrix
cm = confusion_matrix(test_labels, pred_labels)
print("\nConfusion Matrix:")
print(cm)

# Detailed classification report
print("\nDetailed Classification Report:")
print(classification_report(test_labels, pred_labels, target_names=["Non-Sarcastic", "Sarcastic"]))

In [ ]:
# Evaluate on test set
print("Evaluating on test set...\n")
test_results = trainer.evaluate(test_dataset)

print("Test Set Results:")
print(f"  Accuracy:  {test_results['eval_accuracy']:.4f}")
print(f"  Precision: {test_results['eval_precision']:.4f}")
print(f"  Recall:    {test_results['eval_recall']:.4f}")
print(f"  F1-Score:  {test_results['eval_f1']:.4f}")

## 8. Evaluate Model Performance

Evaluate the fine-tuned model on the held-out test set.

In [ ]:
# Train the model
print("Starting fine-tuning...\n")
train_result = trainer.train()

print(f"\n✓ Training completed")
print(f"  Final train loss: {train_result.training_loss:.4f}")
print(f"  Training time: {train_result.training_time_in_s:.0f}s")

## 7. Fine-tune the Model

This will train the model on the training set while monitoring validation performance.
Early stopping will prevent overfitting.

In [ ]:
# Create Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=DataCollatorWithPadding(tokenizer),
    compute_metrics=compute_metrics,
    callbacks=[
        EarlyStoppingCallback(
            early_stopping_patience=2,
            early_stopping_threshold=0.001
        )
    ]
)

print("✓ Trainer created with early stopping (patience=2)")

In [ ]:
# Define compute_metrics function
def compute_metrics(eval_pred):
    """Compute F1, accuracy, precision, recall during training."""
    predictions, labels = eval_pred
    predictions = np.argmax(predictions, axis=1)
    
    return {
        "accuracy": float(accuracy_score(labels, predictions)),
        "precision": float(precision_score(labels, predictions, zero_division=0)),
        "recall": float(recall_score(labels, predictions, zero_division=0)),
        "f1": float(f1_score(labels, predictions, zero_division=0)),
    }

print("✓ Metrics function defined")

## 6. Configure Metrics and Trainer

Define evaluation metrics and create the Trainer with early stopping.

In [ ]:
# Define training arguments
BATCH_SIZE = 8
NUM_EPOCHS = 3
LEARNING_RATE = 3e-5
WARMUP_STEPS = 100
WEIGHT_DECAY = 0.01

training_args = TrainingArguments(
    output_dir="./models/fine_tuned_sarcasm",
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2,
    warmup_steps=WARMUP_STEPS,
    weight_decay=WEIGHT_DECAY,
    learning_rate=LEARNING_RATE,
    logging_dir="./logs",
    logging_steps=10,
    evaluation_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    push_to_hub=False,
    seed=42,
    fp16=torch.cuda.is_available(),  # Use mixed precision on GPU
)

print("Training Configuration:")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Epochs: {NUM_EPOCHS}")
print(f"  Warmup steps: {WARMUP_STEPS}")
print(f"  Weight decay: {WEIGHT_DECAY}")
print(f"  Total training steps: {len(train_dataset) // BATCH_SIZE * NUM_EPOCHS}")
print(f"  Using FP16: {training_args.fp16}")

## 5. Fine-tuning Best Practices for Sarcasm Detection

### Learning Rate Selection
- **Recommended range**: 2e-5 to 5e-5
- **Rationale**: Pre-trained models need small updates to avoid catastrophic forgetting
- **Lower (2e-5)**: More stable, slower convergence
- **Higher (5e-5)**: Faster convergence, risk of divergence

### Warmup Steps
- **Recommended**: 10% of total training steps
- **Purpose**: Gradually increase learning rate to stabilize early training

### Weight Decay (L2 Regularization)
- **Recommended**: 0.01
- **Purpose**: Prevent overfitting on small datasets

### Epochs
- **Recommended**: 3-5 epochs
- **Strategy**: Use early stopping with patience=2

### Domain Shift Considerations
- **Challenge**: Twitter model → Fiction text
- **Solution**: Mixed training, sufficient warmup, conservative learning rate
- **Monitor**: Watch for domain-specific patterns

In [ ]:
# Create HF Dataset objects
def create_hf_dataset(texts, labels, tokenizer, max_length=512):
    """Create Hugging Face Dataset with tokenization."""
    
    def tokenize(examples):
        return tokenizer(
            examples['text'],
            truncation=True,
            max_length=max_length,
            padding=True
        )
    
    # Create dataset
    dataset = Dataset.from_dict({
        'text': texts.tolist(),
        'labels': labels.tolist()
    })
    
    # Tokenize
    dataset = dataset.map(tokenize, batched=True, remove_columns=['text'])
    
    return dataset

print("Creating tokenized datasets...")
train_dataset = create_hf_dataset(train_texts, train_labels, tokenizer)
val_dataset = create_hf_dataset(val_texts, val_labels, tokenizer)
test_dataset = create_hf_dataset(test_texts, test_labels, tokenizer)

print(f"✓ Train dataset: {train_dataset}")
print(f"✓ Val dataset:   {val_dataset}")
print(f"✓ Test dataset:  {test_dataset}")

In [ ]:
# Train/Val/Test split with stratification
print("Creating stratified train/val/test splits...")

labels = df['label'].values
texts = df['text_preprocessed'].values

# First split: 70% train, 30% temp
train_idx, temp_idx = train_test_split(
    range(len(df)),
    train_size=0.7,
    stratify=labels,
    random_state=42
)

# Second split: 50/50 split of temp (15% val, 15% test)
val_proportion = 0.5
val_idx, test_idx = train_test_split(
    temp_idx,
    train_size=val_proportion,
    stratify=labels[temp_idx],
    random_state=42
)

train_texts = texts[train_idx]
train_labels = labels[train_idx]
val_texts = texts[val_idx]
val_labels = labels[val_idx]
test_texts = texts[test_idx]
test_labels = labels[test_idx]

print(f"Train: {len(train_texts)} samples")
print(f"Val:   {len(val_texts)} samples")
print(f"Test:  {len(test_texts)} samples")

print(f"\nClass distribution:")
print(f"Train - Sarcastic: {(train_labels == 1).sum()} ({(train_labels == 1).sum() / len(train_labels) * 100:.1f}%)")
print(f"Val   - Sarcastic: {(val_labels == 1).sum()} ({(val_labels == 1).sum() / len(val_labels) * 100:.1f}%)")
print(f"Test  - Sarcastic: {(test_labels == 1).sum()} ({(test_labels == 1).sum() / len(test_labels) * 100:.1f}%)")

## 4. Prepare Data for Fine-tuning

Split data into train/validation/test with stratification to maintain class balance.
Tokenize using the RoBERTa tokenizer.

In [ ]:
# Load model and tokenizer
MODEL_NAME = "cardiffnlp/twitter-roberta-base-irony"

print(f"Loading model: {MODEL_NAME}")
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

print(f"✓ Model loaded")
print(f"  Architecture: {model.config.model_type}")
print(f"  Hidden size: {model.config.hidden_size}")
print(f"  Number of layers: {model.config.num_hidden_layers}")
print(f"  Number of heads: {model.config.num_attention_heads}")
print(f"  Number of parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

# Check device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)
print(f"  Device: {device}")

## 3. Load Pre-trained Model and Tokenizer

The `cardiffnlp/twitter-roberta-base-irony` model is ideal because:
- Pre-trained on Twitter irony detection (domain similarity)
- RoBERTa handles nuanced linguistic patterns well
- Lightweight (110M params) for fine-tuning with limited data
- Strong performance on English irony/sarcasm detection

In [ ]:
# Apply preprocessing to all texts
print("Preprocessing all texts...")
df['text_preprocessed'] = df['text'].apply(remove_character_names)

print(f"\nSample before/after:")
idx = 0
print(f"Original:     {df['text'].iloc[idx]}")
print(f"Preprocessed: {df['text_preprocessed'].iloc[idx]}")

In [ ]:
# Install spaCy model if needed
import subprocess
import sys

try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    print("Installing spaCy model...")
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

print(f"✓ Loaded spaCy model: en_core_web_sm")

# Create preprocessing function
def remove_character_names(text: str, replacement: str = "[PERSON]") -> str:
    """Remove PERSON entities (character names) from text."""
    doc = nlp(text)
    
    # Track entities to replace (in reverse order to maintain char positions)
    entities_to_replace = [
        (ent.start_char, ent.end_char)
        for ent in doc.ents
        if ent.label_ == "PERSON"
    ]
    
    # Replace from end to start
    result = text
    for start, end in sorted(entities_to_replace, reverse=True):
        result = result[:start] + replacement + result[end:]
    
    return result

# Test preprocessing
test_text = "Sherlock said, 'Oh sure, Watson, that makes perfect sense.'"
preprocessed = remove_character_names(test_text)
print(f"\nOriginal:     {test_text}")
print(f"Preprocessed: {preprocessed}")

## 2. Text Preprocessing with spaCy NER

Remove character names to prevent the model from learning "Character X is always sarcastic" patterns.
This forces the model to focus on linguistic markers instead of character identity.

In [ ]:
# Explore statistics
print("Text length statistics (in characters):")
df['text_length'] = df['text'].str.len()
print(df['text_length'].describe())

print("\n\nSarcastic examples:")
for text in df[df['label'] == 1]['text'].head(3):
    print(f"  • {text}")

print("\n\nSincere examples:")
for text in df[df['label'] == 0]['text'].head(3):
    print(f"  • {text}")

In [ ]:
# Load the Joshi et al. dataset
DATA_PATH = Path("../data/joshi_dataset.json")

# If dataset doesn't exist, provide download instructions
if not DATA_PATH.exists():
    print(f"⚠️  Dataset not found at {DATA_PATH}")
    print("\nDownload instructions:")
    print("1. Clone: git clone https://github.com/amoilanen/joshi-sarcasm-detection.git")
    print("2. Copy data file: cp joshi-sarcasm-detection/data.json data/joshi_dataset.json")
    print("\nFor now, creating dummy dataset for demonstration...")
    
    # Create dummy data for demonstration
    dummy_data = [
        {"text": "Oh sure, because that always works perfectly.", "label": 1},
        {"text": "The weather is beautiful today.", "label": 0},
        {"text": "I absolutely love waiting in traffic.", "label": 1},
        {"text": "This is a sincere statement about reality.", "label": 0},
        {"text": "Yeah right, that makes total sense.", "label": 1},
    ]
else:
    with open(DATA_PATH) as f:
        dummy_data = json.load(f)

print(f"✓ Loaded {len(dummy_data)} samples")
print(f"\nSample structure:")
print(json.dumps(dummy_data[0], indent=2))

# Convert to DataFrame for easier exploration
df = pd.DataFrame(dummy_data)
print(f"\nDataset shape: {df.shape}")
print(f"\nClass distribution:")
print(df['label'].value_counts())
print(f"\nClass proportions:")
print(df['label'].value_counts(normalize=True))

## 1. Load and Explore the Joshi et al. (2016) Book Snippets Dataset

First, download the dataset from: https://github.com/amoilanen/joshi-sarcasm-detection

Expected format: JSON file with list of objects containing `text` and `label` fields.

In [ ]:
# Import Required Libraries
import json
import numpy as np
import pandas as pd
import torch
import spacy
from pathlib import Path
from typing import List, Dict, Tuple

# Hugging Face & PyTorch
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    EarlyStoppingCallback,
    DataCollatorWithPadding
)
from datasets import Dataset, DatasetDict, load_dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Warnings
import warnings
warnings.filterwarnings('ignore')

print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
print(f"Device: {torch.device('cuda' if torch.cuda.is_available() else 'cpu')}")